# 02 - Data Cleaning and Preprocessing

## Objective

The objective of this notebook is to clean and
prepare the original DataCo Supply Chain dataset
for exploratory data analysis.

The process includes:
- Removing unnecessary columns
- Handling missing values
- Converting date columns
- Checking data types
- Validating delivery-related information

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin-1",
    low_memory=False
)

df = df_raw.copy()

print("Original dataset shape:", df_raw.shape)
print("Working dataset shape:", df.shape)

# 2.1 Removing Unnecessary Columns

Some columns contain personal customer information,
while others are empty or have substantial missing
values.

These columns are not required for our current
delivery-performance analysis.

In [ ]:
columns_to_drop = [
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",
    "Customer Street",
    "Customer Zipcode",
    "Order Zipcode",
    "Product Description"
]

df.drop(columns=columns_to_drop, inplace=True)

print("Shape after removing unnecessary columns:", df.shape)

# 2.2 Checking Remaining Missing Values

After removing unnecessary columns, we check
whether any missing values remain in the dataset.

In [ ]:
missing_values = df.isnull().sum()

print(missing_values)

# 2.3 Converting Date Columns

Date columns are converted into Pandas datetime
format so that we can perform time-based analysis,
calculate delivery durations and investigate trends.

In [ ]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

# Just to check if any didn't converted to Date

In [ ]:
print(df[date_columns].isnull().sum())

# To check the data type of the data now 

In [ ]:
print(df.dtypes)

# 2.4 Numerical Data Validation

In this section, we examine numerical columns
to understand their ranges, distributions and
potentially invalid values.

We will investigate shipping duration, sales,
profitability, quantity and other numerical fields.

In [ ]:
print(df.dtypes.to_string())

# 2.5 Examine numerical statistics

In [ ]:
df.describe().T

# 2.6 Check numerical columns individually 

In [ ]:
important_numeric_columns = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Benefit per order",
    "Sales per customer",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Product Price"
]

df[important_numeric_columns].describe().T

# 2.7 Check for invalid numerical values

## Checking Shipping Duration

We examine actual and scheduled shipping days
to identify zero or negative values that may
require further investigation.

In [ ]:
print("Actual shipping days:")
print(df["Days for shipping (real)"].value_counts().sort_index())

print("\nScheduled shipping days:")
print(df["Days for shipment (scheduled)"].value_counts().sort_index())

# 2.8 Check for zero and negative values

In [ ]:
print(
    "Actual shipping days <= 0:",
    (df["Days for shipping (real)"] <= 0).sum()
)

print(
    "Scheduled shipping days <= 0:",
    (df["Days for shipment (scheduled)"] <= 0).sum()
)

# 2.9 Checking for negative financial values 

In [ ]:
financial_columns = [
    "Benefit per order",
    "Sales per customer",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Product Price"
]

for col in financial_columns:
    print(col, ":", (df[col] < 0).sum())

# 2.10 Categorical Data Validation

We examine categorical columns to identify
available categories and unexpected values.

# Order status

In [ ]:
print(df["Order Status"].value_counts())

# Delivery status

In [ ]:
print(df["Delivery Status"].value_counts())

# Delivery Mode 

In [ ]:
print(df["Shipping Mode"].value_counts())

# Product Status 

In [ ]:
print(df["Product Status"].value_counts())

# Check Transaction Types 

In [ ]:
print(df["Type"].value_counts())

# 2.11 Check Actual and scheduled Delivery dates 

In [ ]:
df["shipping_delay_days"] = (
    df["Days for shipping (real)"] -
    df["Days for shipment (scheduled)"]
)

df[[
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "shipping_delay_days",
    "Delivery Status"
]].head(10)

# 2.12 Compare delivery status with late-delivery risk 

In [ ]:
pd.crosstab(
    df["Delivery Status"],
    df["Late_delivery_risk"]
)

# 2.13 Compare delivery status with calculated delay

In [ ]:
pd.crosstab(
    df["Delivery Status"],
    df["shipping_delay_days"] > 0
)

# 2.14 Check whether shipping dates make sense
A shipping date before an order date would be suspicious and worth investigating.

In [ ]:
invalid_date_order = (
    df["shipping date (DateOrders)"] <
    df["order date (DateOrders)"]
)

print("Shipping date before order date:", invalid_date_order.sum())

# 2.15  date difference with the recorded shipping duration

In [ ]:
df["date_shipping_days"] = (
    df["shipping date (DateOrders)"] -
    df["order date (DateOrders)"]
).dt.days

print(
    "Date duration and recorded actual shipping days differ:",
    (
        df["date_shipping_days"] !=
        df["Days for shipping (real)"]
    ).sum()
)

# JUST REVIEWING THE DATA AT LAST

In [ ]:
print("Final working dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nFirst five records:")
display(df.head())

# 2.16 Investigating Date and Shipping Duration Differences

We compare the recorded shipping duration with
the duration calculated from order and shipping
timestamps.

This helps identify differences caused by
date-time recording conventions.

In [ ]:
date_mismatches = df[
    df["date_shipping_days"] != df["Days for shipping (real)"]
]

print("Number of mismatched records:", len(date_mismatches))

display(
    date_mismatches[[
        "order date (DateOrders)",
        "shipping date (DateOrders)",
        "Days for shipping (real)",
        "date_shipping_days"
    ]].head(10)
)

print(
    date_mismatches["Days for shipping (real)"].value_counts()
)

# 2.17 Calculate Exact elapsed hours 

In [ ]:
date_mismatches["elapsed_hours"] = (
    date_mismatches["shipping date (DateOrders)"] -
    date_mismatches["order date (DateOrders)"]
).dt.total_seconds() / 3600

display(
    date_mismatches[[
        "Days for shipping (real)",
        "date_shipping_days",
        "elapsed_hours"
    ]].head(10)
)

# 2.18 Investigate duplicate or redundant columns

In [ ]:
column_pairs = [
    ("Category Id", "Product Category Id"),
    ("Order Item Cardprod Id", "Product Card Id"),
    ("Sales per customer", "Order Item Total"),
    ("Benefit per order", "Order Profit Per Order"),
    ("Product Price", "Order Item Product Price")
]

for col1, col2 in column_pairs:
    print(
        col1,
        "vs",
        col2,
        ":",
        df[col1].equals(df[col2])
    )

# 2.19 Check constant columns

In [ ]:
constant_columns = []

for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        constant_columns.append(col)

print("Constant columns:", constant_columns)

# 2.20 Investigating Order and Delivery Status

We compare order status with delivery status
to understand cancelled, completed and other
order records.

In [ ]:
pd.crosstab(
    df["Order Status"],
    df["Delivery Status"]
)

# 2.21 Counting cancelled delivery records 

In [ ]:
cancelled_deliveries = df[
    df["Delivery Status"] == "Shipping canceled"
]

print("Cancelled delivery records:", len(cancelled_deliveries))

print(
    cancelled_deliveries["Order Status"].value_counts()
)

# 2.22 Calculate delivery-status percentages

In [ ]:
delivery_percentages = (
    df["Delivery Status"]
    .value_counts(normalize=True)
    .mul(100)
)

print(delivery_percentages.round(2))

# 2.23 Final Data Cleaning

Based on our investigation, we remove redundant
and unnecessary columns while preserving useful
business information.

We retain all records, including cancelled
shipments and negative profit values.

In [ ]:
columns_to_remove = [
    "Category Id",
    "Order Item Cardprod Id",
    "Sales per customer",
    "Benefit per order",
    "Product Price",
    "Product Status",
    "date_shipping_days"
]

# Remove the selected columns
df.drop(columns=columns_to_remove, inplace=True)

# Display the updated dataset shape
print("Shape after final cleaning:", df.shape)

# 2.24 Final Data Quality Checks

We perform final checks to verify the cleaned
dataset's dimensions, missing values, duplicates,
identifiers and date columns.

In [ ]:
print("Final dataset shape:", df.shape)

print("\nTotal missing values:", df.isnull().sum().sum())

print("Duplicate rows:", df.duplicated().sum())

print("Unique orders:", df["Order Id"].nunique())

print("Unique order items:", df["Order Item Id"].nunique())

print("\nDate column types:")
print(df[[
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]].dtypes)

print("\nShipping delay column:")
print(df["shipping_delay_days"].describe())

# 2.25 Exporting the Cleaned Dataset

After completing the cleaning and validation
process, we export the final dataset into the
processed data folder.

The original raw dataset remains unchanged.

In [ ]:
import os

# Create the processed folder if it does not exist
os.makedirs("../data/processed", exist_ok=True)

# Save the cleaned dataset
df.to_csv(
    "../data/processed/supply_chain_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

## Final Summary of Data Cleaning

### Original Dataset
- Rows: 180,519
- Columns: 53

### Final Cleaned Dataset
- Rows: 180,519
- Columns: 40

### Cleaning Operations
1. Removed unnecessary customer personal information.
2. Removed columns with excessive missing values.
3. Converted order and shipping dates into datetime format.
4. Investigated numerical values and negative profits.
5. Identified and removed five redundant columns.
6. Removed the constant Product Status column.
7. Removed the temporary date validation column.
8. Created shipping_delay_days to measure the
   difference between actual and scheduled shipping.
9. Retained cancelled shipments and negative profits.
10. Verified missing values and duplicate records.

### Final Result
The cleaned dataset is saved as:
data/processed/supply_chain_cleaned.csv

The original raw dataset remains unchanged.